# CeNN Cellular Attention — reliable Colab benchmark

This notebook benchmarks sparse causal Cellular Attention against the frozen Transformer attention in **SmolLM2-135M**. It now uses a full-pipeline preflight, a fresh process for the scientific run, live stdout/stderr, durable logs, and `last_run.json`, so a child-process failure is never hidden behind only `CalledProcessError`.

The preflight is an engineering check, **not** a scientific result.

## 1 · Fresh checkout and dependencies

In [1]:
import importlib, pathlib, subprocess, sys, tempfile

REPO_REF = "main"
WORK_PARENT = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
REPO_DIR = pathlib.Path(tempfile.mkdtemp(prefix="TinyCeNN-cellular-", dir=WORK_PARENT))

subprocess.run([
    "git", "clone", "--depth", "1", "--branch", REPO_REF,
    "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(REPO_DIR),
    "transformers==4.57.6",
    "datasets>=3,<5",
    "huggingface_hub>=0.34,<2",
    "pandas", "matplotlib", "pytest>=8"
], check=True)

for path in (REPO_DIR, REPO_DIR / "src"):
    sys.path.insert(0, str(path))
importlib.invalidate_caches()

SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Python:", sys.version)
print("Source commit:", SOURCE_COMMIT)
print("Checkout:", REPO_DIR)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Source commit: 4353fb0aed254aca5118bee32d594f9571c3eeb4
Checkout: /content/TinyCeNN-cellular-u_tjmj1m


## 2 · Select GPU and experiment profile

In [2]:
import torch
from pathlib import Path

PROFILE = "balanced"   # smoke | balanced | extended
LAYERS = "18"
SEED = 2026
ALLOW_CPU = False
SAVE_TO_DRIVE = False

VARIANTS = (
    "cellular_local3,cellular_dilated3,cellular_dilated5,"
    "cellular_multiscale5,cellular_shifted8"
)

if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError(
        "GPU not detected. In Colab select Runtime -> Change runtime type -> GPU, then run again."
    )

print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {total:.1f} GiB")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULT_ROOT = Path("/content/drive/MyDrive/TinyCeNN-LM/cellular-attention")
else:
    RESULT_ROOT = WORK_PARENT / "TinyCeNN-cellular-results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print("Profile:", PROFILE)
print("Result root:", RESULT_ROOT)

Device: Tesla T4
GPU memory: 14.6 GiB
Profile: balanced
Result root: /content/TinyCeNN-cellular-results


## 3 · Run repository tests

These tests validate shapes, gradients, strict causality, receptive-field growth, checkpoint reconstruction, and the Llama attention-replacement wrapper before the real-model preflight.

In [3]:
test_command = [
    sys.executable, "-m", "pytest", "-q",
    str(REPO_DIR / "tests/test_cellular_attention.py"),
    str(REPO_DIR / "tests/test_research_layer_benchmark.py"),
]
print("Running:", subprocess.list2cmdline(test_command))
subprocess.run(test_command, cwd=REPO_DIR, check=True)
print("✅ Repository tests passed")

Running: /usr/bin/python3 -m pytest -q /content/TinyCeNN-cellular-u_tjmj1m/tests/test_cellular_attention.py /content/TinyCeNN-cellular-u_tjmj1m/tests/test_research_layer_benchmark.py
✅ Repository tests passed


## 4 · Full-pipeline preflight + scientific run

The launcher first runs one tiny real-model experiment using the actual SmolLM2/FineWeb pipeline. If it succeeds, the selected profile starts in a **new Python process**, which also releases all preflight GPU memory. Output is streamed live and copied to a log file.

In [4]:
import os

runner = REPO_DIR / "scripts/run_cellular_attention_colab.py"
if not runner.exists():
    raise FileNotFoundError(f"Missing launcher: {runner}")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONFAULTHANDLER"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
env["HF_HUB_DISABLE_XET"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

command = [
    sys.executable, "-u", str(runner),
    "--profile", PROFILE,
    "--layers", LAYERS,
    "--variants", VARIANTS,
    "--seed", str(SEED),
    "--output-root", str(RESULT_ROOT),
]
print("Command:", subprocess.list2cmdline(command))
result = subprocess.run(command, cwd=REPO_DIR, env=env)
RUN_RETURN_CODE = result.returncode
print("Launcher exit code:", RUN_RETURN_CODE)

Command: /usr/bin/python3 -u /content/TinyCeNN-cellular-u_tjmj1m/scripts/run_cellular_attention_colab.py --profile balanced --layers 18 --variants cellular_local3,cellular_dilated3,cellular_dilated5,cellular_multiscale5,cellular_shifted8 --seed 2026 --output-root /content/TinyCeNN-cellular-results
Launcher exit code: 1


## 5 · Inspect status and exact failure, if any

In [5]:
import json
import pandas as pd
from IPython.display import display

STATUS_FILE = RESULT_ROOT / "last_run.json"
if not STATUS_FILE.exists():
    raise RuntimeError(f"Launcher did not create {STATUS_FILE}")

status = json.loads(STATUS_FILE.read_text(encoding="utf-8"))
print(json.dumps(status, indent=2))

if status["status"] not in {"completed", "preflight_completed"}:
    print("\n" + "=" * 100)
    print("❌ Benchmark did not complete.")
    print("Stage:", status["status"])
    if status.get("error_tail"):
        print("\nLast log lines:\n")
        print(status["error_tail"])
    print("\nThe partial result directory and full log are preserved.")
    raise RuntimeError(f"Cellular Attention benchmark failed at stage: {status['status']}")

OUTPUT_DIR = Path(status["output_dir"])
LOG_FILE = Path(status["log_file"])
print("\n✅ Benchmark completed")
print("Results:", OUTPUT_DIR)
print("Log:", LOG_FILE)

{
  "status": "preflight_failed",
  "profile": "balanced",
  "layers": "18",
  "variants": "cellular_local3,cellular_dilated3,cellular_dilated5,cellular_multiscale5,cellular_shifted8",
  "seed": 2026,
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "preflight_output_dir": "/content/TinyCeNN-cellular-results/preflight-20260914T200202578785Z",
  "preflight_log": "/content/TinyCeNN-cellular-results/preflight-20260914T200202578785Z.log",
  "preflight_exit_code": 1,
  "error_tail": "Traceback (most recent call last):\n  File \"/content/TinyCeNN-cellular-u_tjmj1m/scripts/benchmark_cellular_attention.py\", line 24, in <module>\n    from scripts.benchmark_cenn_research_layers import (\n    ...<14 lines>...\n    )\nModuleNotFoundError: No module named 'scripts'"
}

❌ Benchmark did not complete.
Stage: preflight_failed

Last log lines:

Traceback (most recent call last):
  File "/content/TinyCeNN-cellular-u_tjmj1m/scripts/benchmark_cellular_attention.py", line 24, in <module>

RuntimeError: Cellular Attention benchmark failed at stage: preflight_failed

## 6 · Decision table

In [ ]:
report = json.loads((OUTPUT_DIR / "cellular_attention_report.json").read_text())
results = pd.read_csv(OUTPUT_DIR / "cellular_attention_summary.csv")
validation = pd.read_csv(OUTPUT_DIR / "validation_summary.csv")
history = pd.read_csv(OUTPUT_DIR / "training_history.csv")

candidates = results[results["variant"] != "transformer_original"].copy()
selected = candidates[candidates["selected_on_validation"].eq(True)].copy()

print("Validation-selected candidates:", report["validation_winners"])
display(selected[[
    "candidate", "context", "test_perplexity", "transformer_perplexity",
    "ppl_ratio", "delta_nll", "delta_nll_ci_low", "delta_nll_ci_high",
    "score_pair_ratio", "receptive_field_tokens", "strict_quality_win", "quality"
]].round(6))

print("\nAll preregistered candidates:")
display(candidates[[
    "variant", "feature_dim", "context", "selected_on_validation",
    "trainable_parameters", "test_perplexity", "ppl_ratio",
    "output_cosine", "grad_mean_cosine", "score_pair_ratio",
    "prefill_speedup", "peak_extra_bytes", "quality"
]].round(6))

## 7 · Quality–sparsity plots

In [ ]:
import matplotlib.pyplot as plt

training_context = {"smoke": 64, "balanced": 256, "extended": 512}[PROFILE]
view = candidates[candidates["context"].eq(training_context)].copy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(view["score_pair_ratio"], view["delta_nll"], s=90)
for _, row in view.iterrows():
    ax.annotate(
        f"{row['variant']} f{int(row['feature_dim'])}",
        (row["score_pair_ratio"], row["delta_nll"]),
        xytext=(5, 5), textcoords="offset points", fontsize=8
    )
ax.axhline(0, linewidth=1)
ax.axhline(0.02, linewidth=1, linestyle="--")
ax.axhline(-0.02, linewidth=1, linestyle="--")
ax.set_xlabel("Sparse score pairs / dense causal Transformer score pairs")
ax.set_ylabel("ΔNLL vs Transformer (lower is better)")
ax.set_title("Cellular Attention: quality–sparsity frontier")
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ordered = view.sort_values("ppl_ratio")
labels = [f"{v}\nf{int(f)}" for v, f in zip(ordered["variant"], ordered["feature_dim"])]
ax.bar(labels, ordered["ppl_ratio"])
ax.axhline(1.0, linewidth=1)
ax.set_ylabel("Perplexity ratio vs Transformer")
ax.set_title("Replacement quality at training context")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 8 · Interpretation

The strongest signal is `strict_quality_win=True`, meaning the complete paired 95% interval for candidate-minus-Transformer NLL is below zero. This remains a **single-layer replacement/adaptation experiment**; it does not yet establish full-model Transformer-free superiority.

## 9 · Download complete results

In [ ]:
import shutil

archive_base = RESULT_ROOT / OUTPUT_DIR.name
archive = shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR)
print("Created:", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running in Google Colab; ZIP remains at:", archive)

In [ ]:
print("Download triggered.")
print("Stopping Colab kernel...")

time.sleep(8)

os.kill(os.getpid(), 9)